In [1]:
!pip install duckdb -q

from google.colab import drive
drive.mount('/content/drive')

import duckdb

BASE = '/content/drive/MyDrive/data-analytics-portfolio/hotel-cancellation-analysis'

con = duckdb.connect()
con.execute(f"""
    CREATE TABLE bookings AS
    SELECT * FROM read_csv_auto('{BASE}/data/clean/clean_bookings.csv')
""")

con.sql("""
    SELECT COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
""").df()

Mounted at /content/drive


,total_bookings,cancellation_rate_pct
0,117399,37.5


In [2]:
con.sql("""
    SELECT hotel,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
    GROUP BY hotel
    ORDER BY cancellation_rate_pct DESC
""").df()

,hotel,total_bookings,cancellation_rate_pct
0,City Hotel,78092,42.2
1,Resort Hotel,39307,28.1


In [3]:
con.sql("""
    SELECT market_segment,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
    GROUP BY market_segment
    ORDER BY cancellation_rate_pct DESC
""").df()

,market_segment,total_bookings,cancellation_rate_pct
0,Undefined,2,100.0
1,Groups,19557,61.7
2,Online TA,56089,36.9
3,Offline TA/TO,23885,34.6
4,Aviation,231,22.1
5,Corporate,5211,18.9
6,Direct,12361,15.4
7,Complementary,63,7.9


In [4]:
con.sql("""
    SELECT market_segment,
           COUNT(*) AS total_bookings,
           SUM(is_canceled) AS canceled_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
    GROUP BY market_segment
    ORDER BY canceled_bookings DESC
""").df()

,market_segment,total_bookings,canceled_bookings,cancellation_rate_pct
0,Online TA,56089,20719.0,36.9
1,Groups,19557,12073.0,61.7
2,Offline TA/TO,23885,8271.0,34.6
3,Direct,12361,1903.0,15.4
4,Corporate,5211,987.0,18.9
5,Aviation,231,51.0,22.1
6,Complementary,63,5.0,7.9
7,Undefined,2,2.0,100.0


In [5]:
con.sql("""
    SELECT lead_time_group,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
    GROUP BY lead_time_group
    ORDER BY MIN(lead_time)
""").df()

,lead_time_group,total_bookings,cancellation_rate_pct
0,0-7 days,18601,9.7
1,8-30 days,18648,28.1
2,31-90 days,29295,37.9
3,91-180 days,26304,44.9
4,180+ days,24551,57.2


In [7]:
con.sql("""
    SELECT market_segment,
           COUNT(*) AS total_bookings,
           SUM(is_canceled) AS canceled_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct,
           ROUND(SUM(is_canceled) * 100.0 / SUM(SUM(is_canceled)) OVER (), 1) AS share_of_cancellations_pct
    FROM bookings
    GROUP BY market_segment
    ORDER BY canceled_bookings DESC
""").df()

,market_segment,total_bookings,canceled_bookings,cancellation_rate_pct,share_of_cancellations_pct
0,Online TA,56089,20719.0,36.9,47.1
1,Groups,19557,12073.0,61.7,27.4
2,Offline TA/TO,23885,8271.0,34.6,18.8
3,Direct,12361,1903.0,15.4,4.3
4,Corporate,5211,987.0,18.9,2.2
5,Aviation,231,51.0,22.1,0.1
6,Complementary,63,5.0,7.9,0.0
7,Undefined,2,2.0,100.0,0.0


In [8]:
con.sql("""
    SELECT lead_time_group,
           COUNT(*) AS total_bookings,
           SUM(is_canceled) AS canceled_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct,
           ROUND(SUM(is_canceled) * 100.0 / SUM(SUM(is_canceled)) OVER (), 1) AS share_of_cancellations_pct
    FROM bookings
    GROUP BY lead_time_group
    ORDER BY MIN(lead_time)
""").df()

,lead_time_group,total_bookings,canceled_bookings,cancellation_rate_pct,share_of_cancellations_pct
0,0-7 days,18601,1803.0,9.7,4.1
1,8-30 days,18648,5248.0,28.1,11.9
2,31-90 days,29295,11113.0,37.9,25.3
3,91-180 days,26304,11802.0,44.9,26.8
4,180+ days,24551,14045.0,57.2,31.9


In [9]:
con.sql("""
    SELECT market_segment,
           ROUND(SUM(CASE WHEN is_canceled = 1 THEN revenue END), 0) AS revenue_lost,
           ROUND(SUM(CASE WHEN is_canceled = 1 THEN revenue END) * 100.0 /
                 SUM(SUM(CASE WHEN is_canceled = 1 THEN revenue END)) OVER (), 1) AS share_of_revenue_lost_pct
    FROM bookings
    GROUP BY market_segment
    ORDER BY revenue_lost DESC
""").df()

,market_segment,revenue_lost,share_of_revenue_lost_pct
0,Online TA,10227646.0,61.1
1,Groups,2800544.0,16.7
2,Offline TA/TO,2492358.0,14.9
3,Direct,993410.0,5.9
4,Corporate,196383.0,1.2
5,Aviation,16578.0,0.1
6,Complementary,270.0,0.0
7,Undefined,48.0,0.0


In [10]:
con.sql("""
    SELECT deposit_type,
           COUNT(*) AS total_bookings,
           SUM(is_canceled) AS canceled_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct,
           ROUND(SUM(is_canceled) * 100.0 / SUM(SUM(is_canceled)) OVER (), 1) AS share_of_cancellations_pct
    FROM bookings
    GROUP BY deposit_type
    ORDER BY canceled_bookings DESC
""").df()

,deposit_type,total_bookings,canceled_bookings,cancellation_rate_pct,share_of_cancellations_pct
0,No Deposit,102650,29481.0,28.7,67.0
1,Non Refund,14587,14494.0,99.4,32.9
2,Refundable,162,36.0,22.2,0.1


In [11]:
con.sql("""
    SELECT market_segment,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
    WHERE deposit_type = 'Non Refund'
    GROUP BY market_segment
    ORDER BY total_bookings DESC
""").df()

,market_segment,total_bookings,cancellation_rate_pct
0,Groups,9172,99.3
1,Offline TA/TO,5006,99.9
2,Corporate,334,93.4
3,Online TA,56,94.6
4,Direct,19,84.2


In [12]:
con.sql("""
    SELECT country,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
    WHERE deposit_type = 'Non Refund'
    GROUP BY country
    ORDER BY total_bookings DESC
    LIMIT 5
""").df()

,country,total_bookings,cancellation_rate_pct
0,PRT,14176,99.7
1,GBR,138,95.7
2,ESP,131,100.0
3,DEU,44,2.3
4,POL,38,97.4


In [13]:
con.sql("""
    SELECT deposit_type,
           ROUND(SUM(CASE WHEN is_canceled = 1 THEN revenue END), 0) AS revenue_lost,
           ROUND(SUM(CASE WHEN is_canceled = 1 THEN revenue END) * 100.0 /
                 SUM(SUM(CASE WHEN is_canceled = 1 THEN revenue END)) OVER (), 1) AS share_of_revenue_lost_pct
    FROM bookings
    GROUP BY deposit_type
    ORDER BY revenue_lost DESC
""").df()

,deposit_type,revenue_lost,share_of_revenue_lost_pct
0,No Deposit,13102902.0,78.3
1,Non Refund,3604337.0,21.5
2,Refundable,19998.0,0.1


In [14]:
con.sql("""
    SELECT MONTH(CAST(arrival_date AS DATE)) AS month_num,
           hotel,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
    GROUP BY month_num, hotel
    ORDER BY month_num, hotel
""").df()

,month_num,hotel,total_bookings,cancellation_rate_pct
0,1,City Hotel,3670,40.2
1,1,Resort Hotel,2130,15.0
2,2,City Hotel,4873,38.7
3,2,Resort Hotel,3046,25.9
4,3,City Hotel,6365,37.4
5,3,Resort Hotel,3272,23.1
6,4,City Hotel,7405,46.6
7,4,Resort Hotel,3547,29.7
8,5,City Hotel,8118,44.9
9,5,Resort Hotel,3490,29.1


In [15]:
con.sql("""
    SELECT country,
           COUNT(*) AS total_bookings,
           SUM(is_canceled) AS canceled_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct,
           ROUND(SUM(is_canceled) * 100.0 / SUM(SUM(is_canceled)) OVER (), 1) AS share_of_cancellations_pct
    FROM bookings
    GROUP BY country
    ORDER BY total_bookings DESC
    LIMIT 10
""").df()

,country,total_bookings,canceled_bookings,cancellation_rate_pct,share_of_cancellations_pct
0,PRT,47027,27332.0,58.1,62.1
1,GBR,12052,2451.0,20.3,5.6
2,FRA,10359,1932.0,18.7,4.4
3,ESP,8488,2176.0,25.6,4.9
4,DEU,7246,1218.0,16.8,2.8
5,ITA,3749,1332.0,35.5,3.0
6,IRL,3369,832.0,24.7,1.9
7,BEL,2334,472.0,20.2,1.1
8,BRA,2210,829.0,37.5,1.9
9,NLD,2099,387.0,18.4,0.9


In [16]:
con.sql("""
    SELECT CASE WHEN country = 'PRT' THEN 'Portugal' ELSE 'Other countries' END AS origin,
           deposit_type,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
    GROUP BY origin, deposit_type
    ORDER BY origin, total_bookings DESC
""").df()

,origin,deposit_type,total_bookings,cancellation_rate_pct
0,Other countries,No Deposit,69847,23.4
1,Other countries,Non Refund,411,86.1
2,Other countries,Refundable,114,11.4
3,Portugal,No Deposit,32803,40.1
4,Portugal,Non Refund,14176,99.7
5,Portugal,Refundable,48,47.9


In [17]:
con.sql("""
    SELECT CASE WHEN total_of_special_requests = 0 THEN '0 requests'
                ELSE '1+ requests' END AS special_requests,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct,
           ROUND(SUM(is_canceled) * 100.0 / SUM(SUM(is_canceled)) OVER (), 1) AS share_of_cancellations_pct
    FROM bookings
    GROUP BY special_requests
    ORDER BY special_requests
""").df()

,special_requests,total_bookings,cancellation_rate_pct,share_of_cancellations_pct
0,0 requests,69124,48.3,75.9
1,1+ requests,48275,22.0,24.1


In [18]:
con.sql("""
    SELECT customer_type,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct,
           ROUND(SUM(is_canceled) * 100.0 / SUM(SUM(is_canceled)) OVER (), 1) AS share_of_cancellations_pct
    FROM bookings
    GROUP BY customer_type
    ORDER BY total_bookings DESC
""").df()

,customer_type,total_bookings,cancellation_rate_pct,share_of_cancellations_pct
0,Transient,88136,41.3,82.6
1,Transient-Party,24668,25.7,14.4
2,Contract,4052,31.1,2.9
3,Group,543,7.7,0.1


In [19]:
con.sql("""
    SELECT CASE WHEN previous_cancellations = 0 THEN '0'
                WHEN previous_cancellations = 1 THEN '1'
                ELSE '2+' END AS previous_cancellations,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
    GROUP BY 1
    ORDER BY 1
""").df()

,previous_cancellations,total_bookings,cancellation_rate_pct
0,0,111008,34.3
1,1,6004,94.7
2,2+,387,57.4


In [20]:
con.sql("""
    SELECT deposit_type,
           COUNT(*) AS total_bookings,
           ROUND(AVG(is_canceled) * 100, 1) AS cancellation_rate_pct
    FROM bookings
    WHERE previous_cancellations = 1
    GROUP BY deposit_type
    ORDER BY total_bookings DESC
""").df()

,deposit_type,total_bookings,cancellation_rate_pct
0,Non Refund,3532,100.0
1,No Deposit,2472,87.1
